<div style="padding: 28px 32px; border-radius: 18px; background: linear-gradient(135deg, #0f172a 0%, #7c3aed 52%, #ec4899 100%); color: white; box-shadow: 0 20px 50px rgba(15, 23, 42, 0.18);">
  <p style="margin: 0; font-size: 14px; letter-spacing: 0.18em; text-transform: uppercase; opacity: 0.85;">Customer Intelligence • Revenue Forecasting System</p>
  <h1 style="margin: 12px 0 10px; font-size: 34px;">Revenue Forecasting</h1>
  <p style="margin: 0; font-size: 17px; max-width: 840px; line-height: 1.7;">
    This notebook aggregates sales into a daily time series and forecasts the next 90 days. Prophet is used when available,
    otherwise the workflow falls back to a robust time-series regression approach.
  </p>
</div>


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.options.display.float_format = "{:,.2f}".format
sns.set_theme(style="whitegrid", context="talk")

project_root = Path.cwd()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from scripts.data_utils import load_cleaned_data
from scripts.forecasting_utils import prepare_daily_sales, generate_revenue_forecast

df = load_cleaned_data(project_root)
daily_sales = prepare_daily_sales(df)
daily_sales.head()


## Daily Revenue Series


In [ ]:
plt.figure(figsize=(16, 5))
plt.plot(daily_sales["order_date"], daily_sales["sales"], color="#7c3aed", linewidth=1.8)
plt.title("Daily Sales Time Series")
plt.xlabel("Order Date")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()


## Forecast the Next 90 Days


In [ ]:
forecast_df, model_name = generate_revenue_forecast(daily_sales, horizon=90)
print(f"Model used: {model_name}")
forecast_df.tail()


In [ ]:
historical_plot = daily_sales.rename(columns={"order_date": "ds", "sales": "actual_sales"})
merged = forecast_df.merge(historical_plot, on="ds", how="left")

plt.figure(figsize=(16, 6))
plt.plot(merged["ds"], merged["yhat"], label="Forecast", color="#ec4899", linewidth=2.5)
plt.fill_between(merged["ds"], merged["yhat_lower"], merged["yhat_upper"], color="#fbcfe8", alpha=0.4, label="Confidence Band")
plt.plot(merged["ds"], merged["actual_sales"], label="Actual Sales", color="#334155", alpha=0.7)
plt.title("Revenue Forecast with Historical Context")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


> **Insight:** Forecasting supports inventory planning, target setting, and stakeholder expectation management. Even a baseline forecast gives the business a more forward-looking operating view than retrospective reporting alone.


## Save Forecast Output


In [ ]:
processed_path = project_root / "data" / "processed" / "forecast_input_daily_sales.csv"
output_path = project_root / "outputs" / "csv" / "revenue_forecast.csv"

processed_path.parent.mkdir(parents=True, exist_ok=True)
output_path.parent.mkdir(parents=True, exist_ok=True)

daily_sales.to_csv(processed_path, index=False)
forecast_df.to_csv(output_path, index=False)

print(f"Saved forecast input to: {processed_path.resolve()}")
print(f"Saved forecast output to: {output_path.resolve()}")
